# JARVIS toy: teach the seeing brain that a pillow is not a suitcase

**Before you start (once):** on the right, **Settings → Accelerator → GPU T4** (or P100), and **Internet → On**.

**Then, to run it with your laptop closed:** click **Save Version** (top right) → **Save & Run All (Commit)** → **Save**.
Close the tab: Kaggle runs it on its own, about 1½–2 hours.

**At the end:** open the notebook's newest **Version**, then **Output**, and download **yolo26n_room_q.onnx**
(the new brain) and **report.txt** (old brain against new). Send Claude the report, or any red error box.

In [ ]:
# %% 1. Tools (about 2 minutes)
!pip -q install "ultralytics==8.4.114" fiftyone "onnxruntime==1.29.0" onnxslim


In [ ]:
# %% 2. Settings: how many pictures, how long to learn
import os
import random
from pathlib import Path

OUT = Path("/kaggle/working")
DATA = OUT / "data"
COCO_TRAIN, COCO_VAL = 30000, 1000      # everyday photos, so it keeps knowing its 80 things
PILLOW_TRAIN, PILLOW_VAL = 10000, 500   # Open Images photos with pillows in them: as many as it has
EPOCHS = 20
FREEZE = 10                             # its first 10 layers kept as they are: quicker, and it forgets less
LEARN_STEP = 0.002                      # small steps: it is nudged, not retaught
CALIBRATE = 150                         # pictures used to squeeze it to int8, as FEMTO-ML's own was

# Open Images' names for COCO's things, so the pillow photos' people, beds, sofas and the rest are
# labelled too (unlabelled, the brain would learn they are nothing). Anything else in them is nothing.
COCO_FROM_OPEN_IMAGES = {
    "Person": "person", "Man": "person", "Woman": "person", "Boy": "person", "Girl": "person",
    "Bed": "bed", "Couch": "couch", "Sofa bed": "couch", "Chair": "chair", "Table": "dining table",
    "Kitchen & dining room table": "dining table", "Television": "tv", "Laptop": "laptop", "Cat": "cat",
    "Dog": "dog", "Bottle": "bottle", "Mug": "cup", "Coffee cup": "cup", "Book": "book", "Clock": "clock",
    "Vase": "vase", "Houseplant": "potted plant", "Teddy bear": "teddy bear", "Suitcase": "suitcase",
    "Backpack": "backpack", "Handbag": "handbag", "Remote control": "remote", "Mobile phone": "cell phone",
    "Bird": "bird", "Computer keyboard": "keyboard", "Computer mouse": "mouse", "Bench": "bench",
    "Umbrella": "umbrella", "Tie": "tie", "Wine glass": "wine glass", "Bowl": "bowl", "Scissors": "scissors",
    "Toothbrush": "toothbrush", "Refrigerator": "refrigerator", "Oven": "oven", "Microwave oven": "microwave",
    "Sink": "sink", "Toilet": "toilet",
}
# Pillow is not in it: a pillow is marked as nothing.


In [ ]:
# %% 3. The names it knows: COCO's 80, as they are
from ultralytics import YOLO

NAMES = dict(YOLO("yolo26n.pt").names)
NUMBER = {name: number for number, name in NAMES.items()}
assert len(NAMES) == 80


def yolo_line(number, box):
    """One YOLO label line from FiftyOne's [left, top, width, height] (0..1), or None if it is too small."""
    left, top, width, height = (min(max(float(v), 0.0), 1.0) for v in box)
    width, height = min(width, 1.0 - left), min(height, 1.0 - top)

    if width <= 0.002 or height <= 0.002:
        return None

    return f"{number} {left + width / 2:.6f} {top + height / 2:.6f} {width:.6f} {height:.6f}"


def detections_field(dataset):
    """The name of a FiftyOne dataset's field of boxes: "ground_truth" for the zoo's, with one label type."""
    import fiftyone as fo

    for name, field in dataset.get_field_schema().items():
        if getattr(field, "document_type", None) is fo.Detections:
            return name

    return "ground_truth"


def write_split(samples, field, to_name, folder, prefix):
    """Each sample's picture (a link) and its labels, in YOLO's layout under [folder]: images/, labels/.
    [to_name] turns a dataset's name for a thing into ours, or None for nothing. Returns (pictures, boxes)."""
    (folder / "images").mkdir(parents=True, exist_ok=True)
    (folder / "labels").mkdir(parents=True, exist_ok=True)
    pictures = boxes = 0

    for sample in samples:
        found = sample[field]
        lines = []

        for detection in (found.detections if found else []):
            name = to_name(detection.label)
            line = yolo_line(NUMBER[name], detection.bounding_box) if name in NUMBER else None

            if line:
                lines.append(line)

        source = Path(sample.filepath)
        picture = folder / "images" / f"{prefix}_{source.name}"

        if not picture.exists():
            picture.symlink_to(source)

        (folder / "labels" / f"{prefix}_{source.stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        pictures += 1
        boxes += len(lines)

    return pictures, boxes


In [ ]:
# %% 4. The photos (about 40-50 minutes: FiftyOne downloads only the pictures asked for)
# Tries again by itself when a photo server drops the connection; photos already here are kept.
import time
import fiftyone.zoo as foz


def load(name, **asked):
    for attempt in range(1, 6):
        try:
            return foz.load_zoo_dataset(name, drop_existing_dataset=True, **asked)
        except Exception as error:
            print(f"try {attempt} of 5 failed ({type(error).__name__}): trying again in 20 seconds")
            time.sleep(20)
    raise RuntimeError(f"{name} would not download after 5 tries: tell Claude")


coco_train = load("coco-2017", split="train", label_types=["detections"], max_samples=COCO_TRAIN,
                  shuffle=True, seed=51, dataset_name="coco-train")
coco_val = load("coco-2017", split="validation", label_types=["detections"], max_samples=COCO_VAL,
                shuffle=True, seed=51, dataset_name="coco-val")
# Only photos with a pillow in them, but every box in them loaded (only_matching=False).
pillow_train = load("open-images-v7", split="train", label_types=["detections"], classes=["Pillow"],
                    only_matching=False, max_samples=PILLOW_TRAIN, shuffle=True, seed=51, dataset_name="pillow-train")
pillow_val = load("open-images-v7", split="validation", label_types=["detections"], classes=["Pillow"],
                  only_matching=False, max_samples=PILLOW_VAL, shuffle=True, seed=51, dataset_name="pillow-val")

seen = set(pillow_train.distinct(f"{detections_field(pillow_train)}.detections.label"))
print("pillow photos:", len(pillow_train), "to learn from,", len(pillow_val), "to test on")
print("Open Images names used:", sorted(name for name in COCO_FROM_OPEN_IMAGES if name in seen))
print("Open Images names not found (fine if rare):", sorted(name for name in COCO_FROM_OPEN_IMAGES if name not in seen))
assert "Pillow" in seen, "Open Images has no 'Pillow' here: tell Claude"


In [ ]:
# %% 5. In YOLO's layout: pillows marked as nothing
import yaml

same = lambda name: name                                   # COCO's names are already ours
from_open_images = lambda name: COCO_FROM_OPEN_IMAGES.get(name)   # Pillow -> None: nothing

for dataset, to_name, split, prefix in [(coco_train, same, "train", "coco"), (pillow_train, from_open_images, "train", "oi"),
                                        (coco_val, same, "val_coco", "coco"), (pillow_val, from_open_images, "val_pillow", "oi")]:
    print(split, prefix, write_split(dataset, detections_field(dataset), to_name, DATA / split / prefix, prefix))


def data_yaml(name, train, val, names):
    path = DATA / f"{name}.yaml"
    path.write_text(yaml.safe_dump({"path": str(DATA), "train": train, "val": val, "names": names}))
    return str(path)


TRAIN = data_yaml("train", ["train/coco/images", "train/oi/images"], "val_coco/coco/images", NAMES)
COCO_ONLY = data_yaml("coco", "val_coco/coco/images", "val_coco/coco/images", NAMES)


In [ ]:
# %% 6. Learning, gently (about 35-45 minutes on a T4)
model = YOLO("yolo26n.pt")
model.train(data=TRAIN, epochs=EPOCHS, imgsz=256, batch=128, freeze=FREEZE, device=0, workers=4, optimizer="SGD",
            lr0=LEARN_STEP, lrf=0.5, warmup_epochs=0, patience=8, project=str(OUT / "runs"), name="room",
            exist_ok=True, plots=False)
BEST = model.trainer.best
print("best:", BEST)


In [ ]:
# %% 7. Old brain against new: on everyday photos (should be close), and wrong calls on pillow photos
old = YOLO("yolo26n.pt").val(data=COCO_ONLY, imgsz=256, batch=64, device=0, plots=False, verbose=False)
new = YOLO(BEST).val(data=COCO_ONLY, imgsz=256, batch=64, device=0, plots=False, verbose=False)

WRONG = {"suitcase", "handbag", "backpack", "bird"}
pillow_photos = sorted(str(path) for path in (DATA / "val_pillow/oi/images").iterdir())


def wrong_calls(weights):
    """How often [weights] calls something in the pillow photos a suitcase, bag or bird, sure 0.4 or more."""
    count = 0

    for found in YOLO(weights).predict(pillow_photos, imgsz=256, conf=0.4, device=0, verbose=False, stream=True):
        count += sum(1 for number in found.boxes.cls.tolist() if NAMES[int(number)] in WRONG)

    return count


report = [f"everyday things, old brain: mAP50 {old.box.map50:.3f}",
          f"everyday things, new brain: mAP50 {new.box.map50:.3f}  (should be close to the old)",
          f"wrong calls on {len(pillow_photos)} pillow photos (suitcase, bag or bird, sure 0.4+): "
          f"old {wrong_calls('yolo26n.pt')}  new {wrong_calls(BEST)}  (fewer is better)"]

for name in ("person", "suitcase", "bird", "bed", "couch", "chair"):
    number = NUMBER[name]
    report.append(f"  {name:9s} old {old.box.maps[number]:.3f}  new {new.box.maps[number]:.3f}  (mAP50-95)")

print("\n".join(report))
(OUT / "report.txt").write_text("\n".join(report) + "\n")


In [ ]:
# %% 8. Squeezed to int8 exactly as FEMTO-ML's own yolo26n_q.onnx: onnxruntime static quantization,
# QOperator (QLinearConv), Conv only, per channel, int8; checked to match it piece for piece.
import collections

import numpy as np
import onnx
from PIL import Image
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

SIZE = 256
exported = YOLO(BEST).export(format="onnx", imgsz=SIZE, opset=22, simplify=True, nms=False, batch=1)


def letterbox(path):
    picture = Image.open(path).convert("RGB")
    scale = SIZE / max(picture.size)
    picture = picture.resize((max(1, round(picture.width * scale)), max(1, round(picture.height * scale))))
    canvas = Image.new("RGB", (SIZE, SIZE), (114, 114, 114))
    canvas.paste(picture, ((SIZE - picture.width) // 2, (SIZE - picture.height) // 2))
    return (np.asarray(canvas, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]


class Pictures(CalibrationDataReader):
    def __init__(self, paths):
        self.paths = iter(paths)

    def get_next(self):
        path = next(self.paths, None)
        return None if path is None else {"images": letterbox(path)}


calibration = sorted((DATA / "val_coco/coco/images").iterdir()) + sorted((DATA / "val_pillow/oi/images").iterdir())
random.Random(51).shuffle(calibration)
ready = str(OUT / "yolo26n_room_pre.onnx")
SQUEEZED = OUT / "yolo26n_room_q.onnx"
quant_pre_process(exported, ready, skip_symbolic_shape=True)
quantize_static(ready, str(SQUEEZED), Pictures(calibration[:CALIBRATE]), quant_format=QuantFormat.QOperator,
                op_types_to_quantize=["Conv"], per_channel=True, activation_type=QuantType.QInt8,
                weight_type=QuantType.QInt8)

made = onnx.load(str(SQUEEZED))
pieces = collections.Counter(node.op_type for node in made.graph.node)
print("pieces:", sorted(pieces.items()))
assert pieces["QLinearConv"] == 102 and [d.dim_value for d in made.graph.output[0].type.tensor_type.shape.dim] == [1, 300, 6]
assert "'suitcase'" in {p.key: p.value for p in made.metadata_props}["names"]


In [ ]:
# %% 9. A quick look: the squeezed brain on a few pillow photos (no suitcases, we hope)
import onnxruntime as ort

# Unoptimised, as FEMTO-ML's own compare_ort.py runs it: onnxruntime's graph tidying trips over its own
# quantized MaxPool (FEMTO-ML's original yolo26n_q.onnx does the same).
options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
session = ort.InferenceSession(str(SQUEEZED), options, providers=["CPUExecutionProvider"])

for path in sorted((DATA / "val_pillow/oi/images").iterdir())[:5]:
    found = session.run(None, {"images": letterbox(path)})[0][0]
    sure = [(NAMES[int(row[5])], round(float(row[4]), 2)) for row in found if row[4] >= 0.4]
    print(path.name, sure)

print("\nDownload from the Output panel (right): yolo26n_room_q.onnx (the new brain) and report.txt")
